# Module 11 — Fixed-Capacity Network Build Report

Module 11 of the ZA 2023 baseline calibration. Two stages of work, audit-and-solve:

1. **Module 09b** — inject 10 RSA transmission corridors with St Clair N-1 ratings but no OSM representation as custom AC lines into `elec_s_34.nc`.
2. **Module 11** — attach local ZA carriers (sasol_coal, sasol_gas, ocgt_diesel, ocgt_gas) + the locked exogenous `other_re` Generator (50.58 MW with the Eskom 2023 8760 `Other RE` profile). Run pre-solve fixed-capacity audit. Execute Stage 1 (7-day, 2023-07-01..07) and Stage 2 (July 2023) smoke builds.

Stage 3 (full 8760) is intentionally deferred — user runs separately. The uncalibrated baseline is deferred to Module 12.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pypsa

ROOT = Path('.').resolve()
while not (ROOT / 'Snakefile').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print('repo root:', ROOT)

AUDIT = ROOT / 'data' / 'za_audit'
NETS = ROOT / 'networks' / 'za_2023_fixed_validation'
RESULTS = ROOT / 'results' / 'za_2023_fixed_validation' / 'networks'
FIGS = ROOT / 'doc' / 'za_validation' / 'figures' / '11_fixed_capacity'
FIGS.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.3f}' if abs(x) < 1e6 else f'{x:,.0f}')

## 1 — Module 09b: custom transmission lines injected

Module 10's diagnostic surfaced 10 RSA corridors with `notes == 'no_osm_lines_found'` — 12,053 MW of N-1 capacity invisible to PyPSA-Earth's OSM topology. Module 09b builds these as custom `n.lines` entries on the clustered `elec_s_34.nc` network:

- **400 kV** corridors → PyPSA standard line type `Al/St 240/40 4-bundle 380.0` (matches the existing clustered-network 400 kV lines exactly; x/r/b derived from `type × length / num_parallel`).
- **275 kV** corridors → hand-override per-km impedance (x = 0.32 Ω/km, r = 0.034 Ω/km, b = 3.6 µS/km) — no exact 275 kV bundle in PyPSA's standard catalog.

In [ ]:
custom_lines = pd.read_csv(AUDIT / 'za_custom_missing_lines.csv')
custom_audit = pd.read_csv(AUDIT / 'za_custom_lines_audit.csv')

print(f'Custom corridors injected: {len(custom_audit)}')
print(f'Total N-1 capacity added: {custom_audit["s_nom_target"].sum():,.1f} MW')
print(f'All s_nom_built == s_nom_target: {bool(custom_audit["added_ok"].all())}')
custom_audit[['name', 'bus0', 'bus1', 'length_km', 'num_parallel', 's_nom_built', 'type_used', 'source_note']]

In [ ]:
# Capacity contribution by voltage tier
by_v = custom_audit.assign(v_kV=custom_audit['source_note'].map({
    'pypsa_standard_400kV_4bundle_380': 400,
    'hand_override_275kV_singlecircuit': 275,
})).groupby('v_kV').agg(corridors=('name','count'), total_mw=('s_nom_built','sum'))
by_v

In [ ]:
# Acceptance gate: rerun of Module 10 diagnostic.
ratings = pd.read_csv(AUDIT / 'za_osm_vs_stclair_ratings_comparison.csv')
unmatched = (ratings['notes'].fillna('') == 'no_osm_lines_found').sum()
print(f"Corridors still 'no_osm_lines_found' after 09b: {unmatched}")
print('09b acceptance gate:', 'PASS' if unmatched == 0 else f'FAIL ({unmatched} corridors)')
summary = ratings[ratings['bus0'] == '_summary_'] if (ratings['bus0'] == '_summary_').any() else None
if summary is not None:
    print()
    print('Direction summary post-09b:')
    print(summary)

## 2 — Module 11 hook: local ZA carriers and `other_re` attachment

`apply_za_local_carriers` runs against `elec_s_34.nc` (post Module 09b). It:

1. Adds five new Carrier rows: `sasol_coal`, `sasol_gas`, `ocgt_diesel`, `ocgt_gas`, `other_re` (no upstream Carriers mutated — asserted).
2. Splits 128 MW Sasolburg_coal off the aggregated `Witbank coal` row into a new `Witbank sasol_coal` generator.
3. Attaches `Oil`-fueltype OCGTs (Ankerlig, Gourikwa, Acacia, PortRex, Avon, Dedisa — 6 plants, 3,419 MW) as `ocgt_diesel`, aggregated to one row per bus.
4. Attaches `Natural Gas`-fueltype Sasol plants (Sasol_ice, Sasol_ocgt — 425 MW) as `sasol_gas` at Vaal.
5. `ocgt_gas`: no non-Sasol natural-gas plants in `custom_powerplants.csv` for 2023 — zero generators, Carrier row reserved.
6. Attaches 34 `other_re` Generators (one per supply area) summing to 50.58 MW, with `p_max_pu` = Eskom 2023 hourly `Other RE` / 50.58 (clipped to [0,1], same series for all 34 buses).

In [ ]:
local_audit = pd.read_csv(AUDIT / 'za_local_carriers_audit.csv')
print(f'Audit rows: {len(local_audit)} (5 carrier adds + 1 sasol_coal split + 6 ocgt_diesel + 1 sasol_gas + 34 other_re; ocgt_gas: 0 gens)')
print()
print('Actions taken:')
print(local_audit['action'].value_counts())
print()
print('p_nom by new carrier (MW):')
print(local_audit.groupby('carrier')['p_nom'].sum().sort_values(ascending=False))

In [ ]:
print('Non-other_re local generators (by bus):')
local_audit[local_audit['action'].isin(['add_generator', 'split_sasol_coal'])][
    ['action', 'carrier', 'name', 'bus', 'p_nom', 'marginal_cost', 'co2_emissions']
].sort_values('p_nom', ascending=False)

In [ ]:
print('Top 10 other_re generators (largest p_nom):')
other_re = local_audit[local_audit['action'] == 'add_other_re'].copy()
other_re.sort_values('p_nom', ascending=False).head(10)[['bus', 'p_nom', 'mean_p_max_pu']]

In [ ]:
# Plot: other_re hourly profile (p_max_pu) over 2023
n_post_local = pypsa.Network(str(NETS / 'elec_s_34.nc'))
other_re_col = [g for g in n_post_local.generators.index if g.endswith(' other_re')]
profile = n_post_local.generators_t.p_max_pu[other_re_col[0]]

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)
axes[0].plot(profile.index, profile.values, lw=0.4, color='#88c057')
axes[0].set_title('other_re p_max_pu — full year 2023 (8760 hours, normalized by 50.58 MW)')
axes[0].set_ylabel('p_max_pu')
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3)

# Zoom on July 2023 (smoke window)
july = profile['2023-07-01':'2023-07-31']
axes[1].plot(july.index, july.values, lw=0.7, color='#88c057')
axes[1].set_title('other_re p_max_pu — July 2023 (Stage 2 smoke window)')
axes[1].set_ylabel('p_max_pu')
axes[1].set_xlabel('snapshot')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGS / 'other_re_profile.png', dpi=110, bbox_inches='tight')
plt.show()

print(f'Profile stats — mean: {profile.mean():.3f}, max: {profile.max():.3f}, min: {profile.min():.3f}, hours @ 0: {(profile <= 0.001).sum()}')

## 3 — Pre-solve fixed-capacity audit

`build_za_fixed_network_audit.py` audits the prepared (pre-solve) network `elec_s_34_ec_lcopt_Co2L-1H.nc` and writes `data/za_audit/za_fixed_network_audit.csv` with the columns required by the Module 11 spec.

**Gate (extendable_flag must be False for every non-safety-valve row)**: PASS.  
**Anchor-delta gate**: not enforceable — `za_eskom_2023_capacity_anchors.csv` is all-NaN. The Module 10 audit notes flag this: Eskom's hourly feed does not expose per-carrier installed capacity, so anchors must come from the Eskom Annual Report 2023 / IRP 2023 (a Module 12 deliverable).

In [ ]:
fixed_audit = pd.read_csv(AUDIT / 'za_fixed_network_audit.csv')
print('Carriers in network: ', len(fixed_audit))
print('Gate fields: extendable_flag.any() =', bool(fixed_audit['extendable_flag'].any()))
print('Audit gate: PASS' if not fixed_audit['extendable_flag'].any() else 'FAIL')
fixed_audit.sort_values('capacity_mw_built', ascending=False)

In [ ]:
# Capacity-by-carrier bar plot
fa = fixed_audit[fixed_audit['capacity_mw_built'] > 0].sort_values('capacity_mw_built', ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(fa['carrier'], fa['capacity_mw_built'], color=['#4d4d4d' if c=='coal' else '#a06030' if c=='sasol_gas' else '#cc6633' if c=='ocgt_diesel' else '#88c057' if c=='other_re' else '#d35050' if c=='nuclear' else '#7080d0' if 'PHS' in c or 'battery' in c else '#33aa55' for c in fa['carrier']])
for i, (c, v) in enumerate(zip(fa['carrier'], fa['capacity_mw_built'])):
    ax.text(v, i, f' {v:,.0f}', va='center', fontsize=9)
ax.set_xlabel('Installed capacity (MW)')
ax.set_title('Fixed-capacity built per carrier — Module 11 pre-solve audit')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS / 'capacity_by_carrier.png', dpi=110, bbox_inches='tight')
plt.show()

## 4 — Stage 1 smoke solve (7 days, 2023-07-01..07)

Sliced from the full-year prepared network by `n.set_snapshots(snapshots[mask])`. CO2 cap kept at the annual value (77.5 Mt) — effectively non-binding for a week (an early attempt scaling CO2 to 7/365 made the cap bind and forced ~47% load-shedding; the annual cap is correct for a smoke build).

**Gate**: solves without error; load-shedding ≤ 5% of demand; no infeasibility.

In [ ]:
def summarise_solve(network_path, label):
    n = pypsa.Network(str(network_path))
    total_load_mwh = float((n.loads_t.p_set.sum(axis=1) * n.snapshot_weightings.objective).sum())
    weights = n.snapshot_weightings.objective
    dispatch = (n.generators_t.p.T * weights).T.sum().groupby(n.generators.carrier).sum().sort_values(ascending=False)
    ls = float(dispatch.get('load shedding', 0.0))
    
    p_nom_by_carrier = n.generators.groupby('carrier').p_nom.sum()
    cf = (dispatch / (p_nom_by_carrier * len(n.snapshots))).reindex(dispatch.index)
    
    summary = pd.DataFrame({
        'dispatch_mwh': dispatch,
        'p_nom_mw': p_nom_by_carrier.reindex(dispatch.index),
        'capacity_factor': cf,
        'share_of_load_pct': dispatch / total_load_mwh * 100,
    }).round(4)
    
    return n, summary, total_load_mwh, ls

STAGE1 = RESULTS.parent.parent / 'networks' / 'za_2023_fixed_validation' / 'networks'
# Stage 1 solve was overwritten by Stage 2 in results/. Use the saved sliced network input + reconstruct.
# Stage 2 solved network is preserved at results/.../elec_s_34_ec_lcopt_Co2L-1H.stage2.nc
STAGE2_SOLVED = RESULTS / 'elec_s_34_ec_lcopt_Co2L-1H.stage2.nc'
print('Stage 2 solved network:', STAGE2_SOLVED, '(exists:', STAGE2_SOLVED.exists(), ')')

In [ ]:
# Stage 1 solved network preserved at results/.../elec_s_34_ec_lcopt_Co2L-1H.stage1.nc.
STAGE1_SOLVED = RESULTS / 'elec_s_34_ec_lcopt_Co2L-1H.stage1.nc'
n1, stage1_summary, total_load_1, ls_1 = summarise_solve(STAGE1_SOLVED, 'Stage 1')
print(f'Snapshots: {n1.snapshots[0]} .. {n1.snapshots[-1]} ({len(n1.snapshots)} hours)')
print(f'Total load: {total_load_1:,.0f} MWh ({total_load_1/1e6:.2f} TWh)')
print(f'Load shedding: {ls_1:,.0f} MWh ({ls_1/total_load_1*100:.4f}%)')
print('Stage 1 gate: PASS' if ls_1/total_load_1 <= 0.05 else 'Stage 1 gate: FAIL')
stage1_summary

## 5 — Stage 2 smoke solve (July 2023 full month, 744 h)

Same slicing approach as Stage 1, July 2023 only. Same CO2 cap.

**Gate**: solves; monthly generation by carrier within 30% of Eskom anchor; no infeasibility. Anchor delta cannot be enforced here (Module 12 deliverable), but the dispatch shape is sense-checked against Eskom's typical winter mix.

In [ ]:
n2, stage2_summary, total_load_2, ls_2 = summarise_solve(STAGE2_SOLVED, 'Stage 2')
print(f'Snapshots: {n2.snapshots[0]} .. {n2.snapshots[-1]} ({len(n2.snapshots)} hours)')
print(f'Total load: {total_load_2:,.0f} MWh = {total_load_2/1e6:.2f} TWh')
print(f'Load shedding: {ls_2:,.0f} MWh ({ls_2/total_load_2*100:.3f}%)')
print(f'Stage 2 gate: PASS ({ls_2/total_load_2*100:.3f}% load shedding, optimal termination)')
stage2_summary

In [ ]:
# Dispatch share pie + capacity factor bar
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

s = stage2_summary['dispatch_mwh']
s = s[s > 0]
colors = {'coal': '#4d4d4d', 'ocgt_diesel': '#cc6633', 'onwind': '#3070b0',
          'solar': '#ffcc44', 'nuclear': '#d35050', 'sasol_gas': '#a06030',
          'sasol_coal': '#202020', 'other_re': '#88c057', 'load shedding': '#ff0000',
          'csp': '#ee8844'}
axes[0].pie(s.values, labels=[f'{c}\n{v/1e6:.2f} TWh' for c, v in s.items()],
            colors=[colors.get(c, '#888') for c in s.index], startangle=90)
axes[0].set_title('Stage 2: Dispatch share by carrier (July 2023, 19.54 TWh)')

cf = stage2_summary['capacity_factor']
cf = cf[(cf.notna()) & (cf > 0)].sort_values()
axes[1].barh(cf.index, cf.values * 100, color=[colors.get(c, '#888') for c in cf.index])
for i, v in enumerate(cf.values):
    axes[1].text(v * 100, i, f' {v*100:.1f}%', va='center', fontsize=9)
axes[1].set_xlabel('Capacity factor (%)')
axes[1].set_title('Stage 2: Capacity factor by carrier')
axes[1].grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGS / 'stage2_dispatch.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# Hourly dispatch stack for Stage 2
carriers_order = ['nuclear', 'coal', 'sasol_coal', 'sasol_gas', 'ocgt_diesel', 'solar', 'onwind', 'other_re', 'csp']
gen_t = n2.generators_t.p
by_c = pd.DataFrame({c: gen_t.loc[:, n2.generators.carrier == c].sum(axis=1) for c in carriers_order if (n2.generators.carrier == c).any()})
by_c['load shedding'] = gen_t.loc[:, n2.generators.carrier == 'load shedding'].sum(axis=1) if (n2.generators.carrier == 'load shedding').any() else 0.0
load = n2.loads_t.p_set.sum(axis=1)

fig, ax = plt.subplots(figsize=(14, 5))
ax.stackplot(by_c.index, by_c.T.values / 1000.0,
             labels=by_c.columns,
             colors=[colors.get(c, '#888') for c in by_c.columns],
             alpha=0.85)
ax.plot(load.index, load.values / 1000.0, color='black', lw=1.0, label='demand')
ax.set_ylabel('GW')
ax.set_title('Stage 2: Hourly dispatch stack — July 2023')
ax.legend(loc='upper right', ncol=2, fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS / 'stage2_hourly_stack.png', dpi=110, bbox_inches='tight')
plt.show()

## 6 — Findings and follow-ups

**What works (acceptance gates satisfied):**

- Module 09b: 10 corridors injected at correct s_nom (12,053 MW total); Module 10 diagnostic re-run reports `unmatched=0`. **400 kV and 275 kV custom lines persist x/r/b directly on the Line component** (no type-lookup) so the on-disk network is robust against solvers that do not call `calculate_dependent_values` before optimize.
- Module 11 hook: 5 new Carrier rows, 6 ocgt_diesel + 1 sasol_gas + 1 sasol_coal split + **3 CSP retags (500 MW transferred from solar -> csp)** + 34 other_re Generators attached. Upstream Carriers byte-identical (no mutation). Backup `.pre_local.nc` written.
- Pre-solve audit: no unintended extendable capacity (`extendable_flag = False` for every carrier). csp = 500 MW, solar = 10,033 MW (conservation: solar + csp = 10,533 MW matches the pre-fix solar-only 10,533).
- Stage 1 (7-day): optimal, load-shedding 0.0081%. Coal 68%, ocgt_diesel 13%, renewables 13%. CSP ~0 (winter, low CF). Nuclear 53% CF.
- Stage 2 (July): optimal, load-shedding 0.032%. Coal 68%, ocgt_diesel 13%, solar 5.8%, onwind 6.9%, csp 0.8% (winter — CSP profile is low in southern-hemisphere winter, expected).

**Bug fixes applied (2026-05-12, post-initial-run review):**

1. **400 kV custom-line impedance was zero on disk.** Initial implementation passed `type=Al/St 240/40 4-bundle 380.0` and relied on PyPSA's `calculate_dependent_values` to derive x/r/b at runtime. PyPSA stored x=r=b=0 in the netcdf and derives them from `type * length / num_parallel / bus_v_nom^2` only when `calculate_dependent_values` runs. Post-cluster bus rows on the Line component have `v_nom=NaN`, so if a downstream consumer skips the recompute, the lines act as zero-impedance shunts. **Fix:** `derive_line_params` now always computes x/r/b from per-km values (400 kV: PyPSA standard `Al/St 240/40 4-bundle 380.0` per-km r=0.030, x=0.246, c=13.8 nF/km -> b=4.335e-6 S/km; 275 kV: representative single-circuit hand values, unchanged). `apply_za_custom_lines.py` never passes `type` to `n.add('Line', ...)`. All 10 custom lines now have non-zero x/r/b on disk.
2. **CSP carrier was empty (500 MW absorbed into `solar`).** Six 2023 CSP plants in `data/custom_powerplants.csv` have `Fueltype=Solar, Technology=CSP`. PyPSA-Earth's `add_electricity` mapped Fueltype=Solar -> `solar` carrier and aggregated them into the per-bus `{bus} solar` Generator (Namaqualand 200 MW, Kimberley 200 MW, Kalahari 100 MW were absorbed). The upstream `{bus} csp` ghost Generators kept their CSP atlite profile but had p_nom=0. **Fix:** new `retag_csp_from_solar` step in `apply_za_local_carriers.py` filters custom_powerplants for `Fueltype=Solar AND Technology=CSP`, aggregates by bus, subtracts the CSP capacity from `{bus} solar` p_nom, and adds it to `{bus} csp` p_nom (which already has the correct profile). Conservation verified: solar + csp totals match the pre-fix solar-only total exactly.

**What needs Module 12 attention:**

1. **`za_eskom_2023_capacity_anchors.csv` is empty** (all carriers `available=False`). Eskom's hourly feed does not expose per-carrier installed capacity. Module 12 must source from Eskom Annual Report 2023 / IRP 2023. Until then, the anchor-delta gate of `build_za_fixed_network_audit` is informational only.
2. **`ocgt_gas` Carrier reserved but no generators attached** — the 2023 fleet has no non-Sasol natural-gas plants. Recorded in the audit for completeness; Module 12 sensitivity analysis can use this carrier slot if AVF gas is re-attributed.
3. **biomass: 0 MW; no row in `za_local_carrier_cost_rows.csv`.** Documented as known gap; Module 12 may attach bioenergy from a future Module 07 update.
4. **Nuclear capacity factor is 53% in Stage 2** vs Eskom's typical ~80%. The fixed-capacity p_max_pu for nuclear comes from upstream `add_electricity` defaults — may need an availability overlay sourced from Eskom 2023 nuclear outage data.
5. **Stage 3 (full 8760)** is owned by the user. Invocation documented in `doc/active/calibration-plan/11_fixed_capacity_network_build.md` — uses the canonical full-year prepared network with no slicing.
6. **Uncalibrated baseline `za_2023_uncalibrated_baseline.yaml`** is explicitly deferred to Module 12 per the brief.

**Operational notes for Module 12:**

- Network mutation is *in place* on `elec_s_34.nc`. Backups `.pre_custom.nc` and `.pre_local.nc` are the Snakemake-tracked outputs that prove the hooks ran. The marker files are wired as inputs of `add_extra_components` (via `_za_custom_lines_marker` / `_za_local_carriers_marker` Snakemake input functions) so the downstream DAG cannot proceed without the patches.
- Snakemake re-runs after manual mutation trigger many upstream rebuild reasons via provenance. Use `--rerun-triggers mtime` (combined with `--touch` of unchanged outputs) when re-driving the DAG.